In [ ]:
import cv2
from pathlib import Path
import torch
from facenet_pytorch import MTCNN
from tqdm import tqdm
import matplotlib.pyplot as plt


In [ ]:
device= "cuda" if torch.cuda.is_available() else "cpu"
print("using device", device)
mtcnn = MTCNN(image_size=224, margin=20, device=device, thresholds=[0.9,0.9,0.9])

In [ ]:
from PIL import Image
raw_dir=Path("../data/raw")
real_videos = list((raw_dir / "original").glob("*.mp4"))
fake_videos=list((raw_dir / "Deepfakes").glob("*.mp4"))

In [ ]:
cap = cv2.VideoCapture(str(real_videos[0]))
ret, frame = cap.read()
cap.release()

frame_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
pil_frame  = Image.fromarray(frame_rgb)

face = mtcnn(pil_frame)

In [ ]:
print(type(face))
print(face.shape if face is not None else "No face detected")

In [ ]:
face_np = face.permute(1,2,0).numpy()
face_np = (face_np - face_np.min()) / (face_np.max() - face_np.min())


In [ ]:
plt.imshow(face_np)
plt.axis("off")
plt.title("Detected face")
plt.show()

plt.imshow(frame_rgb)

In [ ]:
boxes, probs = mtcnn.detect(pil_frame)
print("boxes", boxes)
print("probs", probs)

In [ ]:
filename=Path(real_videos[0].name)
print(filename)

In [ ]:
def get_split(video_num):
    if video_num < 720:
        return "train"
    elif video_num < 860: 
        return "val"
    else:
        return "test"

In [ ]:
from tqdm.notebook import tqdm

def save_faces(videos, label):
    for video in tqdm(videos, desc=f"Processing {label}"):
        cap = None
        try:
            if label == "real":
                video_num = int(Path(video.name).stem)
            else:
                video_num=int(Path(video.name).stem.split("_")[0])
                
            split=get_split(video_num)
            cap = cv2.VideoCapture(str(video))
        
            saved_count=0
            while True:
                ret, frame = cap.read()
                if not ret or saved_count >= 30:
                    break
                frame_rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
                pil_frame=Image.fromarray(frame_rgb)
                face=mtcnn(pil_frame)
                if face is not None:
                    face_np=((face.permute(1,2,0).numpy()+1)/2*255).astype("uint8")
                    face_bgr = cv2.cvtColor(face_np, cv2.COLOR_RGB2BGR)
                    
                    name=f"{Path(video.name).stem}_{saved_count}.jpg"
                    cv2.imwrite(f"../data/processed/{split}/{label}/{name}", face_bgr)
                    saved_count+=1
                for _ in range(9):
                    cap.grab()
        except Exception as e:
            print(f"Error processing {video.name}: {e}")
        finally:
            if cap is not None:
                cap.release()
            
            
            

In [ ]:
save_faces(fake_videos, "fake")

In [ ]:
save_faces(real_videos,"real")

In [ ]:
print(next(mtcnn.parameters()).device)

In [ ]:
import os

real_samples= list(Path("../data/processed/").rglob("real/*.jpg"))[:10]
fake_samples=list(Path("../data/processed/").rglob("fake/*.jpg"))[:10]

fig,axes = plt.subplots(2,4,figsize=(12,6))
for i,ax in enumerate(axes[0]):
    img = cv2.imread(str(real_samples[i]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title("Real")
    ax.axis("off")

plt.show()


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))